# SochDB Local Knowledge Search Walkthrough

This notebook is a clean, local-only demo for the first ICP:
**Python-first AI engineers building local or embedded knowledge retrieval / lightweight RAG systems**.

It shows:
- storing documents in SochDB
- building a local vector index
- retrieving relevant documents
- inspecting the database folder on disk


## 1. Imports and setup

This notebook uses no external model APIs. We create deterministic local embeddings so the whole workflow stays fully offline.

In [1]:
from __future__ import annotations

import hashlib
import json
import os
import shutil
from pathlib import Path

import numpy as np
from sochdb import Database, HnswIndex

In [2]:
DB_PATH = Path('./notebook_knowledge_demo_db')
DIMENSION = 64

DOCUMENTS = [
    {
        'id': 101,
        'title': 'Laptop VPN Setup',
        'body': 'To access internal dashboards, install the company VPN client and connect before opening private services.',
        'tags': ['it', 'security', 'access'],
    },
    {
        'id': 102,
        'title': 'Expense Reimbursement Policy',
        'body': 'Employees should submit travel and meal receipts within 30 days using the finance portal reimbursement form.',
        'tags': ['finance', 'policy', 'travel'],
    },
    {
        'id': 103,
        'title': 'On-Call Incident Process',
        'body': 'If production is degraded, page the on-call engineer, open an incident channel, and post updates every 15 minutes.',
        'tags': ['sre', 'incident', 'operations'],
    },
    {
        'id': 104,
        'title': 'Customer Support Escalation',
        'body': 'Urgent customer issues should be escalated to tier two support with logs, screenshots, and account identifiers attached.',
        'tags': ['support', 'customers', 'triage'],
    },
    {
        'id': 105,
        'title': 'Access Review Checklist',
        'body': 'Managers must review employee access to internal tools quarterly and remove permissions that are no longer required.',
        'tags': ['security', 'access', 'compliance'],
    },
]

if DB_PATH.exists():
    shutil.rmtree(DB_PATH)

## 2. Build deterministic local embeddings

In a production AI system you would usually call an embedding model. For this demo we use a simple hashed embedding so the notebook is reproducible and requires no API keys.

In [3]:
def tokenize(text: str):
    return [
        token.strip('.,:;!?()[]{}').lower()
        for token in text.split()
        if token.strip('.,:;!?()[]{}')
    ]

def embed_text(text: str, dimension: int = DIMENSION) -> np.ndarray:
    vec = np.zeros(dimension, dtype=np.float32)
    for token in tokenize(text):
        digest = hashlib.blake2b(token.encode('utf-8'), digest_size=8).digest()
        bucket = int.from_bytes(digest[:4], 'little') % dimension
        sign = 1.0 if digest[4] % 2 == 0 else -1.0
        vec[bucket] += sign
    norm = np.linalg.norm(vec)
    if norm > 0:
        vec /= norm
    return vec

vectors = np.vstack([
    embed_text(f"{doc['title']} {doc['body']} {' '.join(doc['tags'])}")
    for doc in DOCUMENTS
]).astype(np.float32)
ids = np.array([doc['id'] for doc in DOCUMENTS], dtype=np.uint64)

vectors.shape

(5, 64)

## 3. Store the documents in SochDB

Here SochDB is the local embedded database holding the full document payloads.

In [4]:
db = Database.open(str(DB_PATH))

with db.transaction() as txn:
    for doc in DOCUMENTS:
        key = f"docs/{doc['id']}".encode('utf-8')
        db.put(key, json.dumps(doc).encode('utf-8'), txn.id)

print(f'Stored {len(DOCUMENTS)} documents in {DB_PATH}')

Stored 5 documents in notebook_knowledge_demo_db


## 4. Build a local vector index

The vector index handles similarity search. The database still stores the full document records.

In [5]:
index = HnswIndex(dimension=DIMENSION, m=16, ef_construction=100, metric='cosine')
inserted = index.insert_batch_with_ids(ids, vectors)
print(f'Indexed {inserted} document embeddings')

Indexed 5 document embeddings


## 5. Run a local retrieval query

The query is embedded locally, the vector index returns the nearest IDs, and then we fetch the full documents back from SochDB.

In [6]:
query = 'How do I access internal tools securely from my laptop?'
query_vec = embed_text(query)
result_ids, distances = index.search(query_vec, k=3)

results = []
for doc_id, distance in zip(result_ids.tolist(), distances.tolist()):
    payload = db.get(f'docs/{doc_id}'.encode('utf-8'))
    if payload is None:
        continue
    doc = json.loads(payload.decode('utf-8'))
    results.append({
        'id': doc_id,
        'distance': float(distance),
        'title': doc['title'],
        'body': doc['body'],
    })

print('Query:', query)
results

Query: How do I access internal tools securely from my laptop?


[{'id': 101,
  'distance': 0.6348515748977661,
  'title': 'Laptop VPN Setup',
  'body': 'To access internal dashboards, install the company VPN client and connect before opening private services.'},
 {'id': 105,
  'distance': 0.7160190343856812,
  'title': 'Access Review Checklist',
  'body': 'Managers must review employee access to internal tools quarterly and remove permissions that are no longer required.'},
 {'id': 103,
  'distance': 0.8804771304130554,
  'title': 'On-Call Incident Process',
  'body': 'If production is degraded, page the on-call engineer, open an incident channel, and post updates every 15 minutes.'}]

## 6. Inspect the database folder on disk

SochDB is using a directory-based on-disk layout here. The folder itself is the database root.

In [7]:
sorted(str(path.relative_to(DB_PATH)) for path in DB_PATH.rglob('*'))

['.lock', 'wal.log']

### What those files mean

- `wal.log`: write-ahead log used for durability and recovery
- `.clean_shutdown`: marker indicating the database was closed cleanly

The important idea is:
- the **database API** stores and retrieves records
- the **vector index** finds the nearest document IDs
- the **database folder** is the physical on-disk database root


In [8]:
db.close()
print('Notebook demo complete.')

Notebook demo complete.
